# 강수량 피처 통합 파이프라인
**하수관로 피처에 `rainfall_norm` 추가 (F_SEWER: 9 → 10)**

```
입력  overlap/{split}/sewer_{split}.parquet   F_SEWER=9
출력  overlap/{split}/sewer_{split}.parquet   F_SEWER=10  (rainfall_norm 추가)
```

| Step | 내용 |
|------|------|
| 0 | Imports · 경로 설정 |
| 1 | 매핑 테이블 로드 (sensor_id → rain_station_id) |
| 2 | 강수량 피벗 테이블 함수 정의 |
| 3 | rainfall_norm 조인 함수 정의 |
| 4 | Train / Val / Test 순서로 실행 |
| 5 | gnn_config.json 업데이트 |
| 6 | 최종 검증 |

## Step 0. Imports · 경로 설정

In [ ]:
import gc
import json
import time
from pathlib import Path

import numpy as np
import pandas as pd
import pyarrow as pa
import pyarrow.parquet as pq

PROCESSED    = Path('./dataset/processed')
FEATURES     = Path('./dataset/features')
OVERLAP      = FEATURES / 'overlap'
RAIN_FEAT    = FEATURES / 'rain'
MAPPING_PATH = PROCESSED / 'sewer_rain_mapping.parquet'
GNN_CFG_PATH = FEATURES / 'gnn_config.json'

SPLITS = ('train', 'val', 'test')

print('경로 설정 완료')

## Step 1. 매핑 테이블 로드

`sewer_rain_mapping.parquet` — 하수관로 sensor_id별 최근접 강우관측소 ID 매핑

In [ ]:
df_map = pd.read_parquet(MAPPING_PATH)
print(f'매핑 로드 완료: {MAPPING_PATH}')

# 조회용 딕셔너리
sensor_to_rain = df_map.set_index('sensor_id')['rain_station_id'].to_dict()


## Step 2. 강수량 피벗 테이블 함수

`rain_{split}.parquet` → `(timestamp × station_id)` 피벗 테이블로 변환

```
pivot.loc[timestamp, station_id]  →  rainfall_norm 값을 O(1) 조회
```

sewer timestamp는 분 단위이므로 `floor('10min')` 후 조회.

In [ ]:
def build_rain_pivot(split: str) -> pd.DataFrame:
    path = RAIN_FEAT / split / f'rain_{split}.parquet'
    df   = pd.read_parquet(path, columns=['station_id', 'timestamp', 'rainfall_norm'])
    df['timestamp'] = pd.to_datetime(df['timestamp'])

    pivot = df.pivot_table(
        index='timestamp',
        columns='station_id',
        values='rainfall_norm',
        aggfunc='first',
    ).astype('float32')

    print(f'  [{split}] pivot shape: {pivot.shape[0]:,}행 (타임스텝) × {pivot.shape[1]}열 (관측소)')
    return pivot

## Step 3. rainfall_norm 조인 함수

**조인 방식**: sensor_id 그룹별 벡터화 → row-by-row apply 대비 대폭 빠름

```
for sensor_id in sewer_data:
    rain_station = sensor_to_rain[sensor_id]       # 매핑 1회 조회
    col = pivot[rain_station]                       # 해당 관측소 시계열
    vals = col.reindex(timestamps_floored)          # 타임스탬프 정렬
    → fillna(0.0)  (결측 버킷 = 무강우)
```

In [ ]:
CHUNK_ROWS = 1_000_000   # 청크당 행 수 (메모리 부족 시 줄일 것)

def add_rainfall_norm(split: str, rain_pivot: pd.DataFrame,
                       sensor_to_rain: dict) -> None:
    """sewer_{split}.parquet 에 rainfall_norm 컬럼을 청크 단위로 추가·저장"""
    path     = OVERLAP / split / f'sewer_{split}.parquet'
    tmp_path = path.with_suffix('.tmp.parquet')

    print(f'\n[{split}] 청크 처리 시작...')
    pf = pq.ParquetFile(path)
    total_rows = pf.metadata.num_rows
    print(f'  총 행: {total_rows:,}  |  row groups: {pf.metadata.num_row_groups}')

    # 스키마 확정 (rainfall_norm 중복 방지)
    old_schema = pf.schema_arrow
    if 'rainfall_norm' not in old_schema.names:
        new_schema = old_schema.append(pa.field('rainfall_norm', pa.float32()))
    else:
        new_schema = old_schema

    valid_sta  = set(rain_pivot.columns)
    writer     = None
    processed  = 0
    rain_total = np.float64(0)
    t0 = time.time()

    try:
        for batch in pf.iter_batches(batch_size=CHUNK_ROWS):
            df_c = batch.to_pandas()
            df_c.index = range(len(df_c))   # 청크 내 RangeIndex 보장

            ts10      = df_c['timestamp'].dt.floor('10min')
            rain_norm = np.zeros(len(df_c), dtype='float32')

            for sid, grp in df_c.groupby('sensor_id'):
                sta = sensor_to_rain.get(sid)
                if sta is None or int(sta) not in valid_sta:
                    continue
                vals = (rain_pivot[int(sta)]
                        .reindex(ts10.loc[grp.index])
                        .fillna(0.0)
                        .values
                        .astype('float32'))
                rain_norm[grp.index] = vals

            df_c['rainfall_norm'] = rain_norm
            rain_total += rain_norm.sum()

            table = pa.Table.from_pandas(df_c, schema=new_schema, preserve_index=False)
            if writer is None:
                writer = pq.ParquetWriter(tmp_path, new_schema, compression='snappy')
            writer.write_table(table)

            processed += len(df_c)
            pct = processed / total_rows * 100
            print(f'\r  진행: {processed:,}/{total_rows:,} ({pct:.1f}%)  '
                  f'{time.time()-t0:.0f}s 경과', end='', flush=True)

            del df_c, rain_norm, table
            gc.collect()

    finally:
        if writer is not None:
            writer.close()

    # 임시 파일 → 원본으로 원자적 교체
    tmp_path.replace(path)
    print(f'\n  저장 완료: {path}  ({path.stat().st_size/1e6:.0f} MB)')


def rebuild_sewer_normalized_from_splits() -> None:
    """rainfall_norm이 추가된 split 파일들로 sewer_normalized.parquet를 재구성합니다."""
    out_path = OVERLAP / 'sewer_normalized.parquet'
    tmp_path = out_path.with_suffix('.tmp.parquet')

    writer = None
    first_schema = None
    total_rows = 0

    try:
        for split in SPLITS:
            split_path = OVERLAP / split / f'sewer_{split}.parquet'
            pf = pq.ParquetFile(split_path)
            schema = pf.schema_arrow

            if writer is None:
                first_schema = schema
                writer = pq.ParquetWriter(tmp_path, schema, compression='snappy')
            elif schema != first_schema:
                raise ValueError(f'스키마 불일치: {split_path}')

            for batch in pf.iter_batches(batch_size=CHUNK_ROWS):
                table = pa.Table.from_batches([batch], schema=schema)
                writer.write_table(table)
                total_rows += batch.num_rows

    finally:
        if writer is not None:
            writer.close()

    if writer is None:
        raise RuntimeError('병합할 split 파일이 없습니다.')

    tmp_path.replace(out_path)
    print(f'\nsewer_normalized 재구성 완료: {out_path}  '
          f'({out_path.stat().st_size/1e6:.0f} MB, {total_rows:,}행)')


## Step 4. Train / Val / Test 순서로 실행

각 split마다:
1. 해당 기간 강수량 pivot 빌드
2. 하수관로 데이터에 rainfall_norm 추가·저장

In [ ]:
t_total = time.time()

for split in SPLITS:
    print(f'{"─"*50}')
    print(f'[{split}] 강수량 pivot 빌드...')
    pivot = build_rain_pivot(split)

    add_rainfall_norm(split, pivot, sensor_to_rain)

    del pivot
    gc.collect()

rebuild_sewer_normalized_from_splits()

print(f'\n{"─"*50}')
print(f'전체 소요: {time.time()-t_total:.1f}초')

## Step 5. gnn_config.json 업데이트

- `node_features.sewer` 목록에 `rainfall_norm` 추가 (9 → 10개)
- `future_extension.rainfall.status` : `"placeholder"` → `"active"`

In [ ]:
with open(GNN_CFG_PATH, encoding='utf-8') as f:
    cfg = json.load(f)

# rainfall_norm 추가 (중복 방지)
if 'rainfall_norm' not in cfg['node_features']['sewer']:
    cfg['node_features']['sewer'].append('rainfall_norm')

cfg['future_extension']['rainfall']['status'] = 'active'
cfg['future_extension']['rainfall']['note'] = (
    '서울시 강우관측망 48개소 통합 완료 — 하수관로 노드별 최근접 관측소 매핑'
)

with open(GNN_CFG_PATH, 'w', encoding='utf-8') as f:
    json.dump(cfg, f, ensure_ascii=False, indent=2)

n = len(cfg['node_features']['sewer'])
print(f'sewer features ({n}개): {cfg["node_features"]["sewer"]}')
print(f'rainfall status: {cfg["future_extension"]["rainfall"]["status"]}')

## Step 6. 최종 검증

각 split 파일에 `rainfall_norm` 컬럼이 추가됐는지, 값 범위가 정상인지 확인.

In [ ]:
print('=' * 55)
print('최종 검증')
print('=' * 55)

all_ok = True
for split in SPLITS:
    path = OVERLAP / split / f'sewer_{split}.parquet'
    schema = pq.read_schema(path)

    has_rain = 'rainfall_norm' in schema.names
    mark = 'OK' if has_rain else 'MISSING'
    if not has_rain:
        all_ok = False

    n_col = len(schema.names)
    size = path.stat().st_size / 1e6
    print(f'{mark}  [{split}]  컬럼={n_col}개  {size:.0f}MB')

norm_path = OVERLAP / 'sewer_normalized.parquet'
norm_schema = pq.read_schema(norm_path)
norm_has_rain = 'rainfall_norm' in norm_schema.names
print(f'{"OK" if norm_has_rain else "MISSING"}  sewer_normalized rainfall_norm')
if not norm_has_rain:
    all_ok = False

# gnn_config 확인
with open(GNN_CFG_PATH, encoding='utf-8') as f:
    cfg = json.load(f)
n_feat = len(cfg['node_features']['sewer'])
status = cfg['future_extension']['rainfall']['status']
ok_cfg = (n_feat == 10 and status == 'active')
print(f'{"✅" if ok_cfg else "❌"}  gnn_config.json  '
      f'F_SEWER={n_feat}  status={status}')
if not ok_cfg:
    all_ok = False

print()
print('✅ 강수량 피처 통합 완료' if all_ok else '❌ 미완료 항목 확인 필요')
if all_ok:
    print()
    print('다음 단계 → model_comparison.ipynb')
    print('  커널 재시작 후 셀 1부터 재실행  (F_SEWER 자동으로 10으로 반영됨)')